# YOLOX Training: Combined Dataset (97 Classes) - Quick Validation

Train YOLOX-Tiny on the combined hazmat dataset. **This is a 20-epoch test run** to validate the pipeline before committing to longer training.

## Dataset Composition
- **45 real images** (realImagesExamples) - manually annotated in CVAT
- **217 real images** (realImagesForFineTuning) - previously annotated
- **2000 synthetic images** (label swap synthetic data)
- **Total: 2262 images** (1837 train + 425 val)
- **97 classes** (unified class mapping)

## What to Look For (Signs It's Working)
- Training loss steadily decreasing
- Model detects objects on validation images
- Different classes are predicted (not all same class)
- Bounding boxes roughly in correct locations

## Prerequisites
1. Upload `assets/combined_dataset/` to `/My Drive/HazProML/data/combined_dataset/`
2. Runtime set to **GPU T4** or better

## Estimated Time
- Data copy: 2-5 minutes
- Training: **~1-2 hours** for 20 epochs on T4

## Cell 1: Mount Google Drive & Verify GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU! Enable in Runtime > Change runtime type")

Mounted at /content/drive
PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.7 GB


## Cell 2: Configuration

In [2]:
import os
import json

# ============================================
# PATHS
# ============================================
DRIVE_ROOT = "/content/drive/MyDrive"
DATASET_DIR = f"{DRIVE_ROOT}/HazProML/data/combined_dataset"
CLASS_MAPPING = f"{DATASET_DIR}/class_mapping.json"
DRIVE_OUTPUT = f"{DRIVE_ROOT}/HazProML/models/combined_97class_20epoch_test"
LOCAL_DATA_DIR = '/content/data/combined_dataset'

# ============================================
# TRAINING PARAMETERS
# ============================================
NUM_CLASSES = 97  # Fixed for this dataset
MAX_EPOCHS = 20   # Quick validation run (~1-2 hours on T4)
SAVE_INTERVAL = 5
BATCH_SIZE = 16
INPUT_SIZE = (640, 640)

# ============================================
# SETUP
# ============================================
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Verify class mapping
if os.path.exists(CLASS_MAPPING):
    with open(CLASS_MAPPING, 'r') as f:
        class_map = json.load(f)
    loaded_classes = len(class_map)
    print(f"Loaded class mapping: {loaded_classes} classes")
    if loaded_classes != NUM_CLASSES:
        print(f"WARNING: Expected {NUM_CLASSES} classes but found {loaded_classes}!")
else:
    print(f"WARNING: Class mapping not found at {CLASS_MAPPING}")

# Verify dataset
print(f"\nDataset: {DATASET_DIR}")
print(f"  Exists: {os.path.exists(DATASET_DIR)}")
if os.path.exists(DATASET_DIR):
    train_dir = os.path.join(DATASET_DIR, 'images/train')
    val_dir = os.path.join(DATASET_DIR, 'images/val')
    if os.path.exists(train_dir):
        print(f"  Training images: {len(os.listdir(train_dir))}")
    if os.path.exists(val_dir):
        print(f"  Validation images: {len(os.listdir(val_dir))}")

print(f"\nOutput: {DRIVE_OUTPUT}")
print(f"Classes: {NUM_CLASSES}")
print(f"Epochs: {MAX_EPOCHS} (quick validation run)")
print(f"Checkpoint interval: every {SAVE_INTERVAL} epochs")

Loaded class mapping: 97 classes

Dataset: /content/drive/MyDrive/HazProML/data/combined_dataset
  Exists: True
  Training images: 1837
  Validation images: 425

Output: /content/drive/MyDrive/HazProML/models/combined_97class_20epoch_test
Classes: 97
Epochs: 20 (quick validation run)
Checkpoint interval: every 5 epochs


## Cell 3: Install YOLOX

In [3]:
%%capture
!pip install cython pycocotools thop loguru tabulate

import os
if not os.path.exists('/content/YOLOX'):
    !git clone https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX

%cd /content/YOLOX
!pip install -v -e .

In [4]:
import sys
sys.path.insert(0, '/content/YOLOX')
from yolox.exp import get_exp
print("YOLOX installed successfully!")

YOLOX installed successfully!


## Cell 4: Copy Dataset to Local Storage

Uses `cp -r` for fast copying. Takes 2-5 minutes.

In [5]:
import os

# Clean and create parent directory
!rm -rf /content/data
!mkdir -p /content/data

print("Copying dataset to local storage...")
print(f"Source: {DATASET_DIR}")
print(f"Destination: {LOCAL_DATA_DIR}")
print("\nThis takes 2-5 minutes...\n")

# Use cp -r for fast copying
!cp -r "{DATASET_DIR}" "{LOCAL_DATA_DIR}"

# Verify
train_count = len(os.listdir(f"{LOCAL_DATA_DIR}/images/train"))
val_count = len(os.listdir(f"{LOCAL_DATA_DIR}/images/val"))
print(f"\nDone! {train_count} train images, {val_count} val images")

# Show breakdown by source
print("\nBreakdown by source:")
!ls {LOCAL_DATA_DIR}/images/train | cut -d'_' -f1 | sort | uniq -c

Copying dataset to local storage...
Source: /content/drive/MyDrive/HazProML/data/combined_dataset
Destination: /content/data/combined_dataset

This takes 2-5 minutes...


Done! 1837 train images, 425 val images

Breakdown by source:
    196 real216
     41 real45
   1600 synth


## Cell 5: Convert to COCO Format

YOLOX requires COCO format annotations. This cell converts YOLO format to COCO.

In [6]:
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from datetime import datetime

def yolo_to_coco(images_dir, labels_dir, num_classes, output_path):
    """Convert YOLO annotations to COCO format."""

    # Create categories from class IDs
    # Load class names from class_mapping.json if available
    class_mapping_path = f'{LOCAL_DATA_DIR}/class_mapping.json'
    if os.path.exists(class_mapping_path):
        with open(class_mapping_path, 'r') as f:
            class_map = json.load(f)
        categories = [{"id": int(k), "name": v["name"], "supercategory": "hazmat"}
                      for k, v in class_map.items()]
    else:
        categories = [{"id": i, "name": f"class_{i}", "supercategory": "hazmat"}
                      for i in range(num_classes)]

    images, annotations = [], []
    annotation_id = 0

    # Find all image files (multiple extensions)
    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp', '*.avif']:
        image_files.extend(Path(images_dir).glob(ext))
    image_files = sorted(image_files)

    for img_id, img_path in enumerate(tqdm(image_files, desc="Converting")):
        try:
            with Image.open(img_path) as img:
                width, height = img.size
        except Exception as e:
            print(f"Error reading {img_path}: {e}")
            continue

        images.append({
            "id": img_id, "file_name": img_path.name,
            "width": width, "height": height, "license": 1,
            "date_captured": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

        label_path = Path(labels_dir) / f"{img_path.stem}.txt"
        if not label_path.exists():
            continue

        with open(label_path, 'r') as f:
            content = f.read().strip()
        if not content:
            continue

        for line in content.split('\n'):
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            x_center, y_center, box_w, box_h = map(float, parts[1:])

            # Convert YOLO format (center, w, h) to COCO format (x, y, w, h)
            x = (x_center - box_w / 2) * width
            y = (y_center - box_h / 2) * height
            w = box_w * width
            h = box_h * height

            annotations.append({
                "id": annotation_id, "image_id": img_id, "category_id": class_id,
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2), "iscrowd": 0, "segmentation": []
            })
            annotation_id += 1

    coco = {
        "info": {"description": "HazProML Combined Dataset", "version": "1.0", "year": 2025},
        "licenses": [{"id": 1, "name": "MIT", "url": ""}],
        "categories": categories, "images": images, "annotations": annotations
    }
    with open(output_path, 'w') as f:
        json.dump(coco, f)
    return len(images), len(annotations)

# Create annotations directory
os.makedirs(f'{LOCAL_DATA_DIR}/annotations', exist_ok=True)

# Convert training set
print("Converting training set...")
train_imgs, train_anns = yolo_to_coco(
    f'{LOCAL_DATA_DIR}/images/train',
    f'{LOCAL_DATA_DIR}/labels/train',
    NUM_CLASSES,
    f'{LOCAL_DATA_DIR}/annotations/train.json'
)
print(f"  {train_imgs} images, {train_anns} annotations")

# Convert validation set
print("\nConverting validation set...")
val_imgs, val_anns = yolo_to_coco(
    f'{LOCAL_DATA_DIR}/images/val',
    f'{LOCAL_DATA_DIR}/labels/val',
    NUM_CLASSES,
    f'{LOCAL_DATA_DIR}/annotations/val.json'
)
print(f"  {val_imgs} images, {val_anns} annotations")
print("\nCOCO conversion complete!")

Converting training set...


Converting: 100%|██████████| 1837/1837 [00:00<00:00, 3913.18it/s]


  1837 images, 3665 annotations

Converting validation set...


Converting: 100%|██████████| 425/425 [00:00<00:00, 6690.16it/s]

  425 images, 782 annotations

COCO conversion complete!


## Cell 6: Create YOLOX Config

Creates a custom YOLOX experiment config for 97 classes.

In [7]:
config_content = f'''#!/usr/bin/env python3
import os
import torch
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        # Model: YOLOX-Tiny
        self.depth = 0.33
        self.width = 0.375
        self.num_classes = {NUM_CLASSES}
        self.act = "silu"

        # Data
        self.data_dir = "{LOCAL_DATA_DIR}"
        self.train_ann = "train.json"
        self.val_ann = "val.json"
        self.data_num_workers = 2

        # Training
        self.max_epoch = {MAX_EPOCHS}
        self.warmup_epochs = 5
        self.basic_lr_per_img = 0.01 / 64.0
        self.scheduler = "yoloxwarmcos"
        self.no_aug_epochs = 5  # Disable augmentation for last 5 epochs

        # Input
        self.input_size = {INPUT_SIZE}
        self.test_size = {INPUT_SIZE}
        self.random_size = (14, 26)

        # Augmentation
        self.mosaic_prob = 1.0
        self.mixup_prob = 1.0
        self.enable_mixup = True
        self.flip_prob = 0.5
        self.hsv_prob = 1.0

        # Output
        self.output_dir = "/content/outputs"
        self.exp_name = "yolox_hazmat_97class"
        self.eval_interval = 10  # Evaluate every 10 epochs
        self.print_interval = 50
        self.save_history_ckpt = True

    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=None):
        from yolox.data import COCODataset, TrainTransform, YoloBatchSampler, DataLoader, InfiniteSampler, MosaicDetection, worker_init_reset_seed
        from yolox.utils import wait_for_the_master

        with wait_for_the_master():
            dataset = COCODataset(
                data_dir=self.data_dir,
                json_file=self.train_ann,
                img_size=self.input_size,
                preproc=TrainTransform(max_labels=50, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob),
                cache=False,
                name="images/train"  # Must match directory structure
            )

        dataset = MosaicDetection(
            dataset, mosaic=not no_aug, img_size=self.input_size,
            preproc=TrainTransform(max_labels=120, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob),
            degrees=10.0, translate=0.1, mosaic_scale=(0.1, 2.0), mixup_scale=(0.5, 1.5),
            shear=2.0, enable_mixup=self.enable_mixup, mosaic_prob=self.mosaic_prob, mixup_prob=self.mixup_prob
        )

        self.dataset = dataset
        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(sampler=sampler, batch_size=batch_size, drop_last=False, mosaic=not no_aug)
        return DataLoader(self.dataset, num_workers=self.data_num_workers, pin_memory=True,
                          batch_sampler=batch_sampler, worker_init_fn=worker_init_reset_seed)

    def get_eval_loader(self, batch_size, is_distributed, testdev=False, legacy=False):
        from yolox.data import COCODataset, ValTransform
        from torch.utils.data import DataLoader, SequentialSampler

        valdataset = COCODataset(
            data_dir=self.data_dir,
            json_file=self.val_ann,
            img_size=self.test_size,
            preproc=ValTransform(legacy=legacy),
            name="images/val"  # Must match directory structure
        )
        sampler = SequentialSampler(valdataset)
        return DataLoader(valdataset, batch_size=batch_size, sampler=sampler,
                          num_workers=self.data_num_workers, pin_memory=True)

    def get_evaluator(self, batch_size, is_distributed, testdev=False, legacy=False):
        from yolox.evaluators import COCOEvaluator
        return COCOEvaluator(
            dataloader=self.get_eval_loader(batch_size, is_distributed, testdev, legacy),
            img_size=self.test_size,
            confthre=self.test_conf,
            nmsthre=self.nmsthre,
            num_classes=self.num_classes,
            testdev=testdev
        )
'''

config_path = '/content/YOLOX/exps/hazmat_97class_exp.py'
with open(config_path, 'w') as f:
    f.write(config_content)

print(f"Created config: {config_path}")
print(f"Model: YOLOX-Tiny")
print(f"Classes: {NUM_CLASSES}")
print(f"Epochs: {MAX_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Input size: {INPUT_SIZE}")

Created config: /content/YOLOX/exps/hazmat_97class_exp.py
Model: YOLOX-Tiny
Classes: 97
Epochs: 20
Batch size: 16
Input size: (640, 640)


## Cell 7: Train Model

Main training cell. Features:
- Trains for 20 epochs (quick validation)
- Saves checkpoints every 5 epochs to Google Drive
- Syncs latest_ckpt.pth to Drive every 30s (for resume after disconnect)
- Can resume from existing checkpoint

**Estimated time: ~1-2 hours on T4 GPU**

In [9]:
import os
import shutil
import time
import threading
import re

YOLOX_OUTPUT_DIR = '/content/outputs/yolox_hazmat_97class'
LOG_FILE = f'{YOLOX_OUTPUT_DIR}/train_log.txt'
os.makedirs(YOLOX_OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Download COCO pretrained weights if not present
PRETRAINED_WEIGHTS = '/content/YOLOX/yolox_tiny.pth'
if not os.path.exists(PRETRAINED_WEIGHTS):
  print("Downloading COCO pretrained weights...")
  !wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_tiny.pth -O {PRETRAINED_WEIGHTS}
  print(f"Downloaded: {PRETRAINED_WEIGHTS}")
else:
  print(f"Pretrained weights already exist: {PRETRAINED_WEIGHTS}")

# Background thread to copy checkpoints to Drive
stop_sync = False
saved_epochs = set()
last_logged_epoch = 0

def get_current_epoch():
  """Parse train_log.txt to find current epoch."""
  try:
      if not os.path.exists(LOG_FILE):
          return 0
      with open(LOG_FILE, 'r') as f:
          content = f.read()
      matches = re.findall(r'epoch:\s*(\d+)/\d+', content)
      if matches:
          return int(matches[-1])
  except:
      pass
  return 0

def sync_checkpoints():
  """Background thread that copies checkpoints to Drive."""
  global saved_epochs, last_logged_epoch
  while not stop_sync:
      try:
          current_epoch = get_current_epoch()
          latest_ckpt = os.path.join(YOLOX_OUTPUT_DIR, 'latest_ckpt.pth')

          # Log progress every 10 epochs
          if current_epoch > last_logged_epoch and current_epoch % 10 == 0:
              print(f"\n[Sync] Training at epoch {current_epoch}...")
              last_logged_epoch = current_epoch

          # Always keep latest_ckpt.pth synced to Drive (for resume after disconnect)
          if os.path.exists(latest_ckpt):
              shutil.copy(latest_ckpt, os.path.join(DRIVE_OUTPUT, 'latest_ckpt.pth'))

          # Save named checkpoints at intervals
          for epoch in range(SAVE_INTERVAL, current_epoch + 1, SAVE_INTERVAL):
              if epoch not in saved_epochs:
                  if os.path.exists(latest_ckpt) and current_epoch >= epoch:
                      if current_epoch > epoch or (current_epoch == epoch and os.path.getmtime(latest_ckpt) > time.time() - 60):
                          dst = os.path.join(DRIVE_OUTPUT, f'checkpoint_epoch_{epoch}.pth')
                          shutil.copy(latest_ckpt, dst)
                          saved_epochs.add(epoch)
                          size_mb = os.path.getsize(latest_ckpt) / 1024 / 1024
                          print(f"\n*** Saved checkpoint_epoch_{epoch}.pth to Drive ({size_mb:.1f} MB) ***")
      except Exception as e:
          pass
      time.sleep(30)

# Start background sync thread
sync_thread = threading.Thread(target=sync_checkpoints, daemon=True)
sync_thread.start()

print("="*60)
print(f"TRAINING YOLOX-TINY FOR {MAX_EPOCHS} EPOCHS")
print(f"Dataset: Combined (97 classes, 2262 images)")
print("="*60)
print(f"Output (local): {YOLOX_OUTPUT_DIR}")
print(f"Output (Drive): {DRIVE_OUTPUT}")
print(f"Checkpoints: Saved to Drive every {SAVE_INTERVAL} epochs")
print(f"latest_ckpt.pth synced to Drive every 30s (for resume)")
print("="*60)
print()

%cd /content/YOLOX

# Check if resuming from existing checkpoint
resume_flag = ""
if os.path.exists(os.path.join(YOLOX_OUTPUT_DIR, 'latest_ckpt.pth')):
  resume_flag = "--resume"
  print("Resuming from local checkpoint...")
elif os.path.exists(os.path.join(DRIVE_OUTPUT, 'latest_ckpt.pth')):
  shutil.copy(os.path.join(DRIVE_OUTPUT, 'latest_ckpt.pth'), os.path.join(YOLOX_OUTPUT_DIR, 'latest_ckpt.pth'))
  resume_flag = "--resume"
  print("Restored checkpoint from Drive, resuming...")
else:
  print("Starting training from scratch (COCO pretrained weights)...")

# Run training
!PYTHONPATH=/content/YOLOX python tools/train.py \
  -f exps/hazmat_97class_exp.py \
  -d 1 \
  -b {BATCH_SIZE} \
  --fp16 \
  -o \
  -c {PRETRAINED_WEIGHTS} \
  {resume_flag}

# Stop background sync
stop_sync = True
time.sleep(2)

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

Downloaded: /content/YOLOX/yolox_tiny.pth
TRAINING YOLOX-TINY FOR 20 EPOCHS
Dataset: Combined (97 classes, 2262 images)
Output (local): /content/outputs/yolox_hazmat_97class
Output (Drive): /content/drive/MyDrive/HazProML/models/combined_97class_20epoch_test
Checkpoints: Saved to Drive every 5 epochs
latest_ckpt.pth synced to Drive every 30s (for resume)

/content/YOLOX
Starting training from scratch (COCO pretrained weights)...
2025-12-29 19:06:38.887697: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767035198.907783   10870 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767035198.913793   10870 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W000

In [12]:
# ============================================
# STANDALONE RESUME CELL - FIXED
# ============================================
import os

YOLOX_OUTPUT_DIR = '/content/outputs/yolox_hazmat_97class'
LOCAL_DATA_DIR = '/content/data/combined_dataset'
MAX_EPOCHS = 20
NUM_CLASSES = 97

# Re-create config with evaluation COMPLETELY disabled
config_content = f'''#!/usr/bin/env python3
from yolox.exp import Exp as MyExp
class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth = 0.33
        self.width = 0.375
        self.num_classes = {NUM_CLASSES}
        self.data_dir = "{LOCAL_DATA_DIR}"
        self.train_ann = "train.json"
        self.max_epoch = {MAX_EPOCHS}
        self.warmup_epochs = 5
        self.no_aug_epochs = 5
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.mosaic_prob = 1.0
        self.mixup_prob = 1.0
        self.enable_mixup = True
        self.output_dir = "/content/outputs"
        self.exp_name = "yolox_hazmat_97class"
        self.eval_interval = 99999
        self.print_interval = 50
        self.save_history_ckpt = True
    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=None):
        from yolox.data import COCODataset, TrainTransform, YoloBatchSampler, DataLoader, InfiniteSampler, MosaicDetection, worker_init_reset_seed
        from yolox.utils import wait_for_the_master
        with wait_for_the_master():
            dataset = COCODataset(data_dir=self.data_dir, json_file=self.train_ann, img_size=self.input_size,
                preproc=TrainTransform(max_labels=50, flip_prob=0.5, hsv_prob=1.0), cache=False, name="images/train")
        dataset = MosaicDetection(dataset, mosaic=not no_aug, img_size=self.input_size,
            preproc=TrainTransform(max_labels=120, flip_prob=0.5, hsv_prob=1.0),
            degrees=10.0, translate=0.1, mosaic_scale=(0.1, 2.0), mixup_scale=(0.5, 1.5),
            shear=2.0, enable_mixup=True, mosaic_prob=1.0, mixup_prob=1.0)
        self.dataset = dataset
        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(sampler=sampler, batch_size=batch_size, drop_last=False, mosaic=not no_aug)
        return DataLoader(self.dataset, num_workers=2, pin_memory=True, batch_sampler=batch_sampler, worker_init_fn=worker_init_reset_seed)
    def get_eval_loader(self, *args, **kwargs): return None
    def get_evaluator(self, *args, **kwargs): return None
    def eval(self, model, evaluator, is_distributed, half=False, return_outputs=False):
        # Skip evaluation entirely - return dummy values
        return (0, 0, "Evaluation disabled"), None
'''
with open('/content/YOLOX/exps/hazmat_97class_exp.py', 'w') as f:
    f.write(config_content)
print("Config updated (evaluation FULLY disabled)")

%cd /content/YOLOX
!PYTHONPATH=/content/YOLOX python tools/train.py -f exps/hazmat_97class_exp.py -d 1 -b 16 --fp16 -o --resume

Config updated (evaluation FULLY disabled)
/content/YOLOX
2025-12-29 19:40:40.310754: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767037240.330662   19667 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767037240.336613   19667 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767037240.352011   19667 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767037240.352036   19667 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767037240.352038

## Cell 8: Copy Final Checkpoints to Drive

Run this after training completes to ensure all checkpoints are saved.

In [13]:
import os
import shutil

YOLOX_OUTPUT_DIR = '/content/outputs/yolox_hazmat_97class'

print("Copying final checkpoints to Google Drive...\n")

# Copy epoch checkpoints at intervals
for epoch in range(SAVE_INTERVAL, MAX_EPOCHS + 1, SAVE_INTERVAL):
    src = os.path.join(YOLOX_OUTPUT_DIR, f'epoch_{epoch}_ckpt.pth')
    dst = os.path.join(DRIVE_OUTPUT, f'checkpoint_epoch_{epoch}.pth')
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(src) / 1024 / 1024
        print(f"  checkpoint_epoch_{epoch}.pth ({size_mb:.1f} MB)")

# Copy latest and best checkpoints
latest_src = os.path.join(YOLOX_OUTPUT_DIR, 'latest_ckpt.pth')
best_src = os.path.join(YOLOX_OUTPUT_DIR, 'best_ckpt.pth')

if os.path.exists(latest_src):
    shutil.copy(latest_src, os.path.join(DRIVE_OUTPUT, 'latest_ckpt.pth'))
    print(f"\n  latest_ckpt.pth")

if os.path.exists(best_src):
    shutil.copy(best_src, os.path.join(DRIVE_OUTPUT, 'best_ckpt.pth'))
    print(f"  best_ckpt.pth")

# Copy class mapping for reference
shutil.copy(f'{LOCAL_DATA_DIR}/class_mapping.json', os.path.join(DRIVE_OUTPUT, 'class_mapping.json'))
print(f"  class_mapping.json")

print(f"\nAll checkpoints saved to: {DRIVE_OUTPUT}")
print("\nFiles on Drive:")
for f in sorted(os.listdir(DRIVE_OUTPUT)):
    fpath = os.path.join(DRIVE_OUTPUT, f)
    size = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {f} ({size:.1f} MB)")

Copying final checkpoints to Google Drive...

  checkpoint_epoch_20.pth (38.9 MB)

  latest_ckpt.pth
  class_mapping.json

All checkpoints saved to: /content/drive/MyDrive/HazProML/models/combined_97class_20epoch_test

Files on Drive:
  checkpoint_epoch_10.pth (38.9 MB)
  checkpoint_epoch_20.pth (38.9 MB)
  checkpoint_epoch_5.pth (38.9 MB)
  class_mapping.json (0.0 MB)
  latest_ckpt.pth (38.9 MB)


In [16]:
# ============================================
# RESUME FROM EPOCH 20 → TRAIN TO 50 EPOCHS
# ============================================
import os
import shutil

YOLOX_OUTPUT_DIR = '/content/outputs/yolox_hazmat_97class'
DRIVE_OUTPUT = '/content/drive/MyDrive/HazProML/models/combined_97class_50epoch'
LOCAL_DATA_DIR = '/content/data/combined_dataset'
MAX_EPOCHS = 50
NUM_CLASSES = 97

os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Config with validation disabled
config_content = f'''#!/usr/bin/env python3
from yolox.exp import Exp as MyExp
class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth = 0.33
        self.width = 0.375
        self.num_classes = {NUM_CLASSES}
        self.data_dir = "{LOCAL_DATA_DIR}"
        self.train_ann = "train.json"
        self.max_epoch = {MAX_EPOCHS}
        self.warmup_epochs = 5
        self.no_aug_epochs = 10
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.mosaic_prob = 1.0
        self.mixup_prob = 1.0
        self.enable_mixup = True
        self.output_dir = "/content/outputs"
        self.exp_name = "yolox_hazmat_97class"
        self.eval_interval = 99999
        self.print_interval = 50
        self.save_history_ckpt = True
    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=None):
        from yolox.data import COCODataset, TrainTransform, YoloBatchSampler, DataLoader, InfiniteSampler, MosaicDetection, worker_init_reset_seed
        from yolox.utils import wait_for_the_master
        with wait_for_the_master():
            dataset = COCODataset(data_dir=self.data_dir, json_file=self.train_ann, img_size=self.input_size,
                preproc=TrainTransform(max_labels=50, flip_prob=0.5, hsv_prob=1.0), cache=False, name="images/train")
        dataset = MosaicDetection(dataset, mosaic=not no_aug, img_size=self.input_size,
            preproc=TrainTransform(max_labels=120, flip_prob=0.5, hsv_prob=1.0),
            degrees=10.0, translate=0.1, mosaic_scale=(0.1, 2.0), mixup_scale=(0.5, 1.5),
            shear=2.0, enable_mixup=True, mosaic_prob=1.0, mixup_prob=1.0)
        self.dataset = dataset
        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(sampler=sampler, batch_size=batch_size, drop_last=False, mosaic=not no_aug)
        return DataLoader(self.dataset, num_workers=2, pin_memory=True, batch_sampler=batch_sampler, worker_init_fn=worker_init_reset_seed)
    def get_eval_loader(self, *args, **kwargs): return None
    def get_evaluator(self, *args, **kwargs): return None
    def eval(self, model, evaluator, is_distributed, half=False, return_outputs=False):
        return (0, 0, "Evaluation disabled"), None
'''
with open('/content/YOLOX/exps/hazmat_97class_exp.py', 'w') as f:
    f.write(config_content)
print(f"Config updated: MAX_EPOCHS={MAX_EPOCHS}, no_aug_epochs=10")

# Copy existing checkpoint from 20-epoch run if needed
old_drive = '/content/drive/MyDrive/HazProML/models/combined_97class_20epoch_test/latest_ckpt.pth'
local_ckpt = f'{YOLOX_OUTPUT_DIR}/latest_ckpt.pth'
if not os.path.exists(local_ckpt) and os.path.exists(old_drive):
    shutil.copy(old_drive, local_ckpt)
    print(f"Restored checkpoint from 20-epoch run")

%cd /content/YOLOX
print(f"\nTraining from epoch 20 → {MAX_EPOCHS}...")
print(f"Checkpoints will save to: {DRIVE_OUTPUT}\n")

!PYTHONPATH=/content/YOLOX python tools/train.py -f exps/hazmat_97class_exp.py -d 1 -b 16 --fp16 -o --resume

# Save checkpoints to Drive
print("\n\nSaving checkpoints to Drive...")
for epoch in [25, 30, 35, 40, 45, 50]:
    src = f'{YOLOX_OUTPUT_DIR}/epoch_{epoch}_ckpt.pth'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_OUTPUT}/checkpoint_epoch_{epoch}.pth')
        print(f"  checkpoint_epoch_{epoch}.pth")

shutil.copy(f'{YOLOX_OUTPUT_DIR}/latest_ckpt.pth', f'{DRIVE_OUTPUT}/latest_ckpt.pth')
shutil.copy('/content/data/combined_dataset/class_mapping.json', f'{DRIVE_OUTPUT}/class_mapping.json')
print(f"\nDone! Checkpoints at: {DRIVE_OUTPUT}")

Config updated: MAX_EPOCHS=50, no_aug_epochs=10
/content/YOLOX

Training from epoch 20 → 50...
Checkpoints will save to: /content/drive/MyDrive/HazProML/models/combined_97class_50epoch

2025-12-29 19:56:40.316088: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767038200.335605   23773 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767038200.341443   23773 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767038200.356469   23773 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767038200.356493   23773 computation_placer.cc:177] computation

In [17]:
# ============================================
# COMPARE INFERENCE ACROSS CHECKPOINTS
# ============================================
import json, cv2, numpy as np, torch
from pathlib import Path
import matplotlib.pyplot as plt
from yolox.exp import get_exp
from yolox.utils import postprocess
from yolox.data.data_augment import ValTransform

CONF_THRESH = 0.10
CHECKPOINTS = [40, 45, 50]  # Epochs to compare
YOLOX_OUTPUT_DIR = '/content/outputs/yolox_hazmat_97class'
LOCAL_DATA_DIR = '/content/data/combined_dataset'

# Load class names
with open(f'{LOCAL_DATA_DIR}/class_mapping.json', 'r') as f:
    class_map = json.load(f)
CLASSES = [class_map[str(i)]['name'] for i in range(len(class_map))]
COLORS = np.random.randint(0, 255, size=(len(CLASSES), 3), dtype=np.uint8)
preproc = ValTransform(legacy=False)

# Get validation images (prefer real images)
val_dir = Path(f'{LOCAL_DATA_DIR}/images/val')
real_images = sorted(val_dir.glob('real_*'))[:4]
if len(real_images) < 4:
    real_images.extend(sorted(val_dir.glob('synth_*'))[:4-len(real_images)])
val_images = real_images[:4]

fig, axes = plt.subplots(len(CHECKPOINTS), 4, figsize=(16, 4*len(CHECKPOINTS)))

for row, epoch in enumerate(CHECKPOINTS):
    # Load checkpoint
    ckpt_path = f'{YOLOX_OUTPUT_DIR}/epoch_{epoch}_ckpt.pth'
    if not os.path.exists(ckpt_path):
        ckpt_path = f'{YOLOX_OUTPUT_DIR}/latest_ckpt.pth'
        print(f"Epoch {epoch} not found, using latest")

    exp = get_exp('/content/YOLOX/exps/hazmat_97class_exp.py', None)
    model = exp.get_model()
    ckpt = torch.load(ckpt_path, map_location='cuda')
    model.load_state_dict(ckpt['model'])
    model.cuda().eval()

    for col, img_path in enumerate(val_images):
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        tensor, _ = preproc(img, None, (640, 640))
        tensor = torch.from_numpy(tensor).unsqueeze(0).float().cuda()

        with torch.no_grad():
            out = model(tensor)
            out = postprocess(out, len(CLASSES), CONF_THRESH, 0.45)

        num_det = 0
        if out[0] is not None:
            det = out[0].cpu().numpy()
            scale = min(640/h, 640/w)
            for box, score, cls_id in zip(det[:,:4]/scale, det[:,4]*det[:,5], det[:,6].astype(int)):
                x0,y0,x1,y1 = map(int, box)
                color = tuple(map(int, COLORS[cls_id]))
                cv2.rectangle(img, (x0,y0), (x1,y1), color, 2)
                cv2.putText(img, f"{CLASSES[cls_id][:10]}:{score:.2f}", (x0,y0-5), cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
            num_det = len(det)

        ax = axes[row, col] if len(CHECKPOINTS) > 1 else axes[col]
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Epoch {epoch}: {num_det} det")
        ax.axis('off')

    print(f"Epoch {epoch}: tested")

plt.suptitle(f'Checkpoint Comparison (conf={CONF_THRESH})', fontsize=14)
plt.tight_layout()
plt.savefig('/content/checkpoint_comparison.png', dpi=150)
plt.show()

Output hidden; open in https://colab.research.google.com to view.

## Cell 9: Test Inference (Optional)

Run inference on validation images to visually verify the model.

In [18]:
import json
import cv2
import numpy as np
import torch
from pathlib import Path
import matplotlib.pyplot as plt

from yolox.exp import get_exp
from yolox.utils import postprocess
from yolox.data.data_augment import ValTransform

# Load model
ckpt_path = os.path.join(DRIVE_OUTPUT, 'latest_ckpt.pth')
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(YOLOX_OUTPUT_DIR, 'latest_ckpt.pth')

exp = get_exp('/content/YOLOX/exps/hazmat_97class_exp.py', None)
model = exp.get_model()
ckpt = torch.load(ckpt_path, map_location='cuda')
model.load_state_dict(ckpt['model'])
model.cuda().eval()

print(f"Model loaded from epoch {ckpt.get('start_epoch', '?')}")

# Load class names
with open(f'{LOCAL_DATA_DIR}/class_mapping.json', 'r') as f:
    class_map = json.load(f)
CLASSES = [class_map[str(i)]['name'] for i in range(len(class_map))]

preproc = ValTransform(legacy=False)
COLORS = np.random.randint(0, 255, size=(len(CLASSES), 3), dtype=np.uint8)

# Get validation images (prefer real images)
val_dir = Path(f'{LOCAL_DATA_DIR}/images/val')
real_images = list(val_dir.glob('real_*'))
if len(real_images) < 8:
    real_images.extend(list(val_dir.glob('synth_*'))[:8-len(real_images)])
val_images = real_images[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, img_path in zip(axes.flat, val_images):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]

    tensor, _ = preproc(img, None, (640, 640))
    tensor = torch.from_numpy(tensor).unsqueeze(0).float().cuda()

    with torch.no_grad():
        out = model(tensor)
        out = postprocess(out, len(CLASSES), 0.25, 0.45)

    if out[0] is not None:
        det = out[0].cpu().numpy()
        scale = min(640/h, 640/w)
        boxes = det[:, :4] / scale
        scores = det[:, 4] * det[:, 5]
        cls_ids = det[:, 6].astype(int)

        for box, score, cls_id in zip(boxes, scores, cls_ids):
            if cls_id >= len(CLASSES):
                continue
            x0, y0, x1, y1 = map(int, box)
            color = tuple(map(int, COLORS[cls_id % len(COLORS)]))
            cv2.rectangle(img, (x0, y0), (x1, y1), color, 2)
            label = f"{CLASSES[cls_id][:15]}:{score:.2f}"
            cv2.putText(img, label, (x0, y0-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        num_det = len(boxes)
    else:
        num_det = 0

    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{img_path.name[:20]}... ({num_det} det)")
    ax.axis('off')

plt.suptitle('Model Inference on Validation Images', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT, 'inference_results.png'), dpi=150)
plt.show()
print(f"\nSaved inference results to {DRIVE_OUTPUT}/inference_results.png")

Output hidden; open in https://colab.research.google.com to view.

## Done! (20-Epoch Validation Run)

### Checkpoints saved to Google Drive:
- `checkpoint_epoch_5.pth`, `checkpoint_epoch_10.pth`, `checkpoint_epoch_15.pth`, `checkpoint_epoch_20.pth`
- `latest_ckpt.pth` (final model)
- `class_mapping.json` (97 class definitions)

### Location:
`/My Drive/HazProML/models/combined_97class_20epoch_test/`

### How to Know If It's Working:
1. **Loss decreased** during training (check the training output)
2. **Cell 9 shows detections** on real images with correct labels
3. **Different classes** are predicted (not all the same)
4. **Bounding boxes** are roughly in the right locations

### If Validation Passes:
1. Change `MAX_EPOCHS = 20` to `MAX_EPOCHS = 100` in Cell 2
2. Update `DRIVE_OUTPUT` path to `combined_97class_100epoch`
3. Re-run from Cell 6 onwards for full training

### If Validation Fails:
- Check training loss - is it decreasing or stuck?
- Check class distribution in annotations
- Verify COCO conversion worked (Cell 5 output)
- Check that images loaded correctly (Cell 4 output)

In [19]:
import json, cv2, numpy as np, torch
from pathlib import Path
import matplotlib.pyplot as plt
from yolox.exp import get_exp
from yolox.utils import postprocess
from yolox.data.data_augment import ValTransform

CONF_THRESH = 0.10  # Lower threshold to catch more detections

exp = get_exp('/content/YOLOX/exps/hazmat_97class_exp.py', None)
model = exp.get_model()
ckpt = torch.load('/content/outputs/yolox_hazmat_97class/latest_ckpt.pth', map_location='cuda')
model.load_state_dict(ckpt['model'])
model.cuda().eval()

with open('/content/data/combined_dataset/class_mapping.json', 'r') as f:
    class_map = json.load(f)
CLASSES = [class_map[str(i)]['name'] for i in range(len(class_map))]
preproc = ValTransform(legacy=False)
COLORS = np.random.randint(0, 255, size=(len(CLASSES), 3), dtype=np.uint8)

val_dir = Path('/content/data/combined_dataset/images/val')
val_images = list(val_dir.glob('real_*'))[:8] or list(val_dir.glob('*'))[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_path in zip(axes.flat, val_images):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    tensor, _ = preproc(img, None, (640, 640))
    tensor = torch.from_numpy(tensor).unsqueeze(0).float().cuda()
    with torch.no_grad():
        out = model(tensor)
        out = postprocess(out, len(CLASSES), CONF_THRESH, 0.45)
    num_det = 0
    if out[0] is not None:
        det = out[0].cpu().numpy()
        scale = min(640/h, 640/w)
        for box, score, cls_id in zip(det[:,:4]/scale, det[:,4]*det[:,5], det[:,6].astype(int)):
            x0,y0,x1,y1 = map(int, box)
            color = tuple(map(int, COLORS[cls_id]))
            cv2.rectangle(img, (x0,y0), (x1,y1), color, 2)
            cv2.putText(img, f"{CLASSES[cls_id][:12]}:{score:.2f}", (x0,y0-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        num_det = len(det)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{img_path.name[:20]}... ({num_det} det)")
    ax.axis('off')
plt.suptitle(f'Inference @ conf={CONF_THRESH}', fontsize=14)
plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [21]:
# ============================================
# RESUME FROM EPOCH 50 → TRAIN TO 100 EPOCHS
# ============================================
import os
import shutil

YOLOX_OUTPUT_DIR = '/content/outputs/yolox_hazmat_97class'
DRIVE_OUTPUT = '/content/drive/MyDrive/HazProML/models/combined_97class_100epoch'
LOCAL_DATA_DIR = '/content/data/combined_dataset'
MAX_EPOCHS = 100
NUM_CLASSES = 97

os.makedirs(DRIVE_OUTPUT, exist_ok=True)

config_content = f'''#!/usr/bin/env python3
from yolox.exp import Exp as MyExp
class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth = 0.33
        self.width = 0.375
        self.num_classes = {NUM_CLASSES}
        self.data_dir = "{LOCAL_DATA_DIR}"
        self.train_ann = "train.json"
        self.max_epoch = {MAX_EPOCHS}
        self.warmup_epochs = 5
        self.no_aug_epochs = 15
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.mosaic_prob = 1.0
        self.mixup_prob = 1.0
        self.enable_mixup = True
        self.output_dir = "/content/outputs"
        self.exp_name = "yolox_hazmat_97class"
        self.eval_interval = 99999
        self.print_interval = 50
        self.save_history_ckpt = True
    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=None):
        from yolox.data import COCODataset, TrainTransform, YoloBatchSampler, DataLoader, InfiniteSampler, MosaicDetection, worker_init_reset_seed
        from yolox.utils import wait_for_the_master
        with wait_for_the_master():
            dataset = COCODataset(data_dir=self.data_dir, json_file=self.train_ann, img_size=self.input_size,
                preproc=TrainTransform(max_labels=50, flip_prob=0.5, hsv_prob=1.0), cache=False, name="images/train")
        dataset = MosaicDetection(dataset, mosaic=not no_aug, img_size=self.input_size,
            preproc=TrainTransform(max_labels=120, flip_prob=0.5, hsv_prob=1.0),
            degrees=10.0, translate=0.1, mosaic_scale=(0.1, 2.0), mixup_scale=(0.5, 1.5),
            shear=2.0, enable_mixup=True, mosaic_prob=1.0, mixup_prob=1.0)
        self.dataset = dataset
        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(sampler=sampler, batch_size=batch_size, drop_last=False, mosaic=not no_aug)
        return DataLoader(self.dataset, num_workers=2, pin_memory=True, batch_sampler=batch_sampler, worker_init_fn=worker_init_reset_seed)
    def get_eval_loader(self, *args, **kwargs): return None
    def get_evaluator(self, *args, **kwargs): return None
    def eval(self, model, evaluator, is_distributed, half=False, return_outputs=False):
        return (0, 0, "Evaluation disabled"), None
'''
with open('/content/YOLOX/exps/hazmat_97class_exp.py', 'w') as f:
    f.write(config_content)
print(f"Config updated: MAX_EPOCHS={MAX_EPOCHS}")

old_drive = '/content/drive/MyDrive/HazProML/models/combined_97class_50epoch/latest_ckpt.pth'
local_ckpt = f'{YOLOX_OUTPUT_DIR}/latest_ckpt.pth'
if not os.path.exists(local_ckpt) and os.path.exists(old_drive):
    shutil.copy(old_drive, local_ckpt)
    print(f"Restored checkpoint from 50-epoch run")

%cd /content/YOLOX
print(f"\nTraining from epoch 50 → {MAX_EPOCHS}...")

!PYTHONPATH=/content/YOLOX python tools/train.py -f exps/hazmat_97class_exp.py -d 1 -b 16 --fp16 -o --resume

# Save checkpoints every 5 epochs
print("\n\nSaving checkpoints to Drive...")
for epoch in range(55, 101, 5):  # 55, 60, 65, 70, 75, 80, 85, 90, 95, 100
    src = f'{YOLOX_OUTPUT_DIR}/epoch_{epoch}_ckpt.pth'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_OUTPUT}/checkpoint_epoch_{epoch}.pth')
        print(f"  checkpoint_epoch_{epoch}.pth")
shutil.copy(f'{YOLOX_OUTPUT_DIR}/latest_ckpt.pth', f'{DRIVE_OUTPUT}/latest_ckpt.pth')
shutil.copy('/content/data/combined_dataset/class_mapping.json', f'{DRIVE_OUTPUT}/class_mapping.json')
print(f"\nDone! Checkpoints at: {DRIVE_OUTPUT}")

Config updated: MAX_EPOCHS=100
/content/YOLOX

Training from epoch 50 → 100...
2025-12-29 20:55:13.929051: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767041713.948522   38611 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767041713.954541   38611 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767041713.969663   38611 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767041713.969689   38611 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00